
PINN to solve the 3D Maxwell Inertial Landau-Lifshitz-Gilbert (MILLG) equation



In [ ]:
import torch
import torch.nn as nn
import numpy as np

def build_model(input_dim, hidden_layers, output_dim):
    """
    Build a fully connected neural network model with given architecture.
    
    Args:
        input_dim (int): Number of input dimensions (e.g., 4 for t, x, y, z in 3D MILLG)
        hidden_layers (list): List containing the number of neurons in each hidden layer
        output_dim (int): Number of output dimensions (e.g., 9 for E, H, M components)
        
    Returns:
        torch.nn.Sequential: Neural network model
    """
    # Create a list to hold all layers
    layers = []
    
    # Input layer
    layers.append(nn.Linear(input_dim, hidden_layers[0]))
    layers.append(nn.Tanh())
    
    # Hidden layers
    for i in range(len(hidden_layers) - 1):
        layers.append(nn.Linear(hidden_layers[i], hidden_layers[i + 1]))
        layers.append(nn.Tanh())
    
    # Output layer
    layers.append(nn.Linear(hidden_layers[-1], output_dim))
    
    # Create the model
    model = nn.Sequential(*layers)
    
    # Initialize weights with Xavier initialization
    for layer in model:
        if isinstance(layer, nn.Linear):
            nn.init.xavier_normal_(layer.weight)
            nn.init.zeros_(layer.bias)
            
    return model

def get_derivative(output, input_tensor, order=1):
    """
    Compute the derivative of the output with respect to input_tensor using autograd.
    
    Args:
        output (torch.Tensor): Output tensor to differentiate (e.g., neural network predictions)
        input_tensor (torch.Tensor): Input tensor to differentiate with respect to (e.g., t, x, y, z)
        order (int): Order of differentiation (1 for first derivative, 2 for second, etc.)
        
    Returns:
        torch.Tensor: Derivative of output with respect to input_tensor
    """
    # First-order derivative
    grad_outputs = torch.ones_like(output)
    grad = torch.autograd.grad(output, input_tensor, grad_outputs=grad_outputs, create_graph=True, allow_unused=True)[0]
    
    # Higher-order derivatives
    for _ in range(1, order):
        if grad is None:
            return None
        grad_outputs = torch.ones_like(grad)
        grad = torch.autograd.grad(grad, input_tensor, grad_outputs=grad_outputs, create_graph=True, allow_unused=True)[0]
    
    return grad



In [ ]:
# -*- coding: utf-8 -*-

import torch
import numpy as np
from pyDOE import lhs
import utilities
import matplotlib.pyplot as plt
import vtk
from vtkmodules.util import numpy_support
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dtype = torch.float
torch.manual_seed(10)
np.random.seed(10)

# ### Parameters
N_f = 2000  # PDE collocation points
N_ic = 200  # Initial condition points
T = 0.125    # Time domain [0, T]
domain_bounds = [-1, 2]  # Spatial domain [-1, 2]^3
material_bounds = [0, 1]  # Material cube [0, 1]^3

# ### Physical Constants with Normalization
# Original values
mu0_orig = 4 * np.pi * 1e-7
epsilon_orig = 8.854e-12
gamma_orig = 2.21e5  # Gyromagnetic ratio
alpha = 0.1          # Damping parameter
A_orig = 1e-11       # Exchange constant
Ms_orig = 1e6        # Saturation magnetization

# Normalized values for numerical stability
Hnorm = 1.0
Mnorm = 1.0
Enorm = 1.0
Lnorm = 1.0  # Length normalization
Tnorm = 1.0  # Time normalization

# Normalized constants
mu0 = 1.0
epsilon = 1.0
gamma = 1.0
Ms = 1.0  # Normalize to unit magnitude
A = 0.1   # Normalized exchange constant
omega_m = 1e-2  # Inertial parameter (normalized)

# ### Generate Collocation Points
lb = np.array([0] + [domain_bounds[0]] * 3)
ub = np.array([T] + [domain_bounds[1]] * 3)
X_f = lb + (ub - lb) * lhs(4, N_f)  # [t, x, y, z]

lb_ic = np.array([domain_bounds[0]] * 3)
ub_ic = np.array([domain_bounds[1]] * 3)
X_ic = lb_ic + (ub_ic - lb_ic) * lhs(3, N_ic)  # [x, y, z] at t=0
X_ic = np.hstack((np.zeros((N_ic, 1)), X_ic))

# Convert to tensors
t_f = torch.tensor(X_f[:, 0:1], dtype=dtype, requires_grad=True, device=device)
x_f = torch.tensor(X_f[:, 1:2], dtype=dtype, requires_grad=True, device=device)
y_f = torch.tensor(X_f[:, 2:3], dtype=dtype, requires_grad=True, device=device)
z_f = torch.tensor(X_f[:, 3:4], dtype=dtype, requires_grad=True, device=device)

t_ic = torch.tensor(X_ic[:, 0:1], dtype=dtype, requires_grad=True, device=device)
x_ic = torch.tensor(X_ic[:, 1:2], dtype=dtype, requires_grad=True, device=device)
y_ic = torch.tensor(X_ic[:, 2:3], dtype=dtype, requires_grad=True, device=device)
z_ic = torch.tensor(X_ic[:, 3:4], dtype=dtype, requires_grad=True, device=device)

# ### Material Indicator
def is_inside(x, y, z):
    return (material_bounds[0] <= x) & (x <= material_bounds[1]) & \
           (material_bounds[0] <= y) & (y <= material_bounds[1]) & \
           (material_bounds[0] <= z) & (z <= material_bounds[1])

inside_f = is_inside(X_f[:, 1], X_f[:, 2], X_f[:, 3]).astype(float)
inside_ic = is_inside(X_ic[:, 1], X_ic[:, 2], X_ic[:, 3]).astype(float)
inside_f_tensor = torch.tensor(inside_f, dtype=dtype, device=device).reshape(-1, 1)

# ### Initial Conditions
def initial_E(x, y, z):
    # Using a Gaussian pulse as initial condition for E_x
    sigma = 0.2
    amplitude = 0.1
    x0, y0, z0 = 0.5, 0.5, 0.5  # Center of the pulse
    
    Ex = amplitude * np.exp(-((x-x0)**2 + (y-y0)**2 + (z-z0)**2) / (2*sigma**2))
    Ey = np.zeros_like(x)
    Ez = np.zeros_like(x)
    return Ex, Ey, Ez

def initial_H(x, y, z):
    return np.zeros_like(x), np.zeros_like(x), np.ones_like(x)

import numpy as np

def initial_M(x, y, z, inside_f):
    # Center of the material (assuming coordinates are normalized to [0, 1])
    x_center, y_center, z_center = 0.5, 0.5, 0.5
    
    # Distance from the center
    dx = x - x_center
    dy = y - y_center
    dz = z - z_center
    r = np.sqrt(dx**2 + dy**2 + dy**2)  # Radial distance
    
    # Compute magnetization components (normalized to magnitude 1)
    mx = inside_f * np.exp(-dx**2 / r)
    my = inside_f * np.exp(-dy**2 / r)
    mz = inside_f * np.exp(-1 / r)
    
    return mx, my, mz

E0x_ic, E0y_ic, E0z_ic = initial_E(X_ic[:, 1], X_ic[:, 2], X_ic[:, 3])
H0x_ic, H0y_ic, H0z_ic = initial_H(X_ic[:, 1], X_ic[:, 2], X_ic[:, 3])
M0x_ic, M0y_ic, M0z_ic = initial_M(X_ic[:, 1], X_ic[:, 2], X_ic[:, 3], inside_ic)

E0_ic = torch.tensor(np.stack((E0x_ic, E0y_ic, E0z_ic), axis=1), dtype=dtype, device=device)
H0_ic = torch.tensor(np.stack((H0x_ic, H0y_ic, H0z_ic), axis=1), dtype=dtype, device=device)
M0_ic = torch.tensor(np.stack((M0x_ic, M0y_ic, M0z_ic), axis=1), dtype=dtype, device=device)

# ### Physics-Informed MILLG Model
class PhysicsInformedMILLGModel:
    def __init__(self, t_f, x_f, y_f, z_f, t_ic, x_ic, y_ic, z_ic, inside_f, E0_ic, H0_ic, M0_ic):
        self.t_f = t_f
        self.x_f = x_f
        self.y_f = y_f
        self.z_f = z_f
        self.t_ic = t_ic
        self.x_ic = x_ic
        self.y_ic = y_ic
        self.z_ic = z_ic
        self.inside_f = inside_f
        self.E0_ic = E0_ic
        self.H0_ic = H0_ic
        self.M0_ic = M0_ic
        
        # Deeper network for better expressivity
        self.model = utilities.build_model(4, [128, 128, 256, 256, 128, 128], 9).to(device)
        self.optimizer = None
        self.loss_history = {'total': [], 'pde': [], 'ic': [], 'M_out': [], 'M_magnitude': []}
        
        # Initialize model with proper scaling
        self.initialize_weights()
    
    def initialize_weights(self):
        # Xavier initialization is already done in utilities.build_model
        pass
    
    def get_fields(self, t, x, y, z):
        input_tensor = torch.cat((t, x, y, z), dim=1).to(device)
        output = self.model(input_tensor)
        E = output[:, 0:3]  # [E_x, E_y, E_z]
        H = output[:, 3:6]  # [H_x, H_y, H_z]
        M = output[:, 6:9]  # [M_x, M_y, M_z]
        
        # Normalize M to ensure its magnitude is constrained to 1
        M_magnitude = torch.sqrt(torch.sum(M**2, dim=1, keepdim=True))
        M = M / (M_magnitude + 1e-8)  # Add small epsilon to avoid division by zero
        
        return E, H, M
    
    def compute_derivatives(self, field, t, x, y, z):
        # First time derivative
        ft = utilities.get_derivative(field, t, 1)
        # Second time derivative
        ftt = utilities.get_derivative(field, t, 2) if ft is not None else None
        # First spatial derivatives
        fx = utilities.get_derivative(field, x, 1)
        fy = utilities.get_derivative(field, y, 1)
        fz = utilities.get_derivative(field, z, 1)
        # Second spatial derivatives for Laplacian
        fxx = utilities.get_derivative(field, x, 2)
        fyy = utilities.get_derivative(field, y, 2)
        fzz = utilities.get_derivative(field, z, 2)
        return ft, ftt, fx, fy, fz, fxx, fyy, fzz
    
    def cost_function(self):
        # PDE residuals
        E, H, M = self.get_fields(self.t_f, self.x_f, self.y_f, self.z_f)
        
        # Derivatives
        Ex_t, _, Ex_x, Ex_y, Ex_z, _, _, _ = self.compute_derivatives(E[:, 0:1], self.t_f, self.x_f, self.y_f, self.z_f)
        Ey_t, _, Ey_x, Ey_y, Ey_z, _, _, _ = self.compute_derivatives(E[:, 1:2], self.t_f, self.x_f, self.y_f, self.z_f)
        Ez_t, _, Ez_x, Ez_y, Ez_z, _, _, _ = self.compute_derivatives(E[:, 2:3], self.t_f, self.x_f, self.y_f, self.z_f)
        Hx_t, _, Hx_x, Hx_y, Hx_z, _, _, _ = self.compute_derivatives(H[:, 0:1], self.t_f, self.x_f, self.y_f, self.z_f)
        Hy_t, _, Hy_x, Hy_y, Hy_z, _, _, _ = self.compute_derivatives(H[:, 1:2], self.t_f, self.x_f, self.y_f, self.z_f)
        Hz_t, _, Hz_x, Hz_y, Hz_z, _, _, _ = self.compute_derivatives(H[:, 2:3], self.t_f, self.x_f, self.y_f, self.z_f)
        Mx_t, Mx_tt, Mx_x, Mx_y, Mx_z, Mx_xx, Mx_yy, Mx_zz = self.compute_derivatives(M[:, 0:1], self.t_f, self.x_f, self.y_f, self.z_f)
        My_t, My_tt, My_x, My_y, My_z, My_xx, My_yy, My_zz = self.compute_derivatives(M[:, 1:2], self.t_f, self.x_f, self.y_f, self.z_f)
        Mz_t, Mz_tt, Mz_x, Mz_y, Mz_z, Mz_xx, Mz_yy, Mz_zz = self.compute_derivatives(M[:, 2:3], self.t_f, self.x_f, self.y_f, self.z_f)
        
        # Curl of E and H
        curl_E_x = Ez_y - Ey_z
        curl_E_y = Ex_z - Ez_x
        curl_E_z = Ey_x - Ex_y
        curl_H_x = Hz_y - Hy_z
        curl_H_y = Hx_z - Hz_x
        curl_H_z = Hy_x - Hx_y
        
        # Laplacian of M
        lap_Mx = Mx_xx + Mx_yy + Mx_zz
        lap_My = My_xx + My_yy + My_zz
        lap_Mz = Mz_xx + Mz_yy + Mz_zz
        
        # Effective field
        H_eff_x = H[:, 0:1] + A * lap_Mx
        H_eff_y = H[:, 1:2] + A * lap_My
        H_eff_z = H[:, 2:3] + A * lap_Mz
        
        # Maxwell residuals
        R1_x = curl_E_x + mu0 * (Hx_t + self.inside_f * Mx_t)
        R1_y = curl_E_y + mu0 * (Hy_t + self.inside_f * My_t)
        R1_z = curl_E_z + mu0 * (Hz_t + self.inside_f * Mz_t)
        R2_x = curl_H_x - epsilon * Ex_t
        R2_y = curl_H_y - epsilon * Ey_t
        R2_z = curl_H_z - epsilon * Ez_t
        
        # LLG terms
        # M × H_eff
        M_cross_Heff_x = M[:, 1:2] * H_eff_z - M[:, 2:3] * H_eff_y
        M_cross_Heff_y = M[:, 2:3] * H_eff_x - M[:, 0:1] * H_eff_z
        M_cross_Heff_z = M[:, 0:1] * H_eff_y - M[:, 1:2] * H_eff_x
        
        # M × dM/dt
        M_cross_Mt_x = M[:, 1:2] * Mz_t - M[:, 2:3] * My_t
        M_cross_Mt_y = M[:, 2:3] * Mx_t - M[:, 0:1] * Mz_t
        M_cross_Mt_z = M[:, 0:1] * My_t - M[:, 1:2] * Mx_t
        
        # Inertial term: Ωₘ × (∂²M/∂t²)
        M_cross_Mtt_x = M[:, 1:2] * Mz_tt - M[:, 2:3] * My_tt
        M_cross_Mtt_y = M[:, 2:3] * Mx_tt - M[:, 0:1] * Mz_tt
        M_cross_Mtt_z = M[:, 0:1] * My_tt - M[:, 1:2] * Mx_tt
        
        # LLG residuals with inertia
        R3_x = Mx_t + gamma * M_cross_Heff_x - alpha * M_cross_Mt_x - omega_m * M_cross_Mtt_x
        R3_y = My_t + gamma * M_cross_Heff_y - alpha * M_cross_Mt_y - omega_m * M_cross_Mtt_y
        R3_z = Mz_t + gamma * M_cross_Heff_z - alpha * M_cross_Mt_z - omega_m * M_cross_Mtt_z
        
        # PDE loss - Maxwell equations
        loss_maxwell = torch.mean(R1_x**2 + R1_y**2 + R1_z**2) + torch.mean(R2_x**2 + R2_y**2 + R2_z**2)
        
        # PDE loss - LLG equation (only inside material)
        loss_llg = torch.mean(self.inside_f * (R3_x**2 + R3_y**2 + R3_z**2))
        
        # Combined PDE loss
        loss_pde = loss_maxwell + loss_llg
        
        # Enforce M = 0 outside material
        M_outside = (1 - self.inside_f) * (M[:, 0:1]**2 + M[:, 1:2]**2 + M[:, 2:3]**2)
        loss_M_out = torch.mean(M_outside)
        
        # Enforce |M| = Ms inside material
        M_magnitude = torch.sqrt(M[:, 0:1]**2 + M[:, 1:2]**2 + M[:, 2:3]**2)
        M_magnitude_error = self.inside_f * ((M_magnitude - Ms)**2)
        loss_M_magnitude = torch.mean(M_magnitude_error)
        
        # Initial condition loss
        E_ic, H_ic, M_ic = self.get_fields(self.t_ic, self.x_ic, self.y_ic, self.z_ic)
        loss_ic = (torch.mean((E_ic - self.E0_ic)**2) +
                   torch.mean((H_ic - self.H0_ic)**2) +
                   torch.mean((M_ic - self.M0_ic)**2))
        
        # Weights to balance different physics components
        w_pde = 1.0
        w_M_out = 1.0
        w_ic = 1.0
        w_M_magnitude = 1.0
        
        # Total loss
        # total_loss = w_pde * loss_pde + w_M_out * loss_M_out + w_ic * loss_ic + w_M_magnitude * loss_M_magnitude
        total_loss = w_pde * loss_pde + w_ic * loss_ic
        # Store individual losses for tracking
        self.current_losses = {
            'total': total_loss.item(),
            'pde': loss_pde.item(),
            'ic': loss_ic.item(),
            'M_out': loss_M_out.item(),
            'M_magnitude': loss_M_magnitude.item()
        }
        
        return total_loss
    
    def closure(self):
        self.optimizer.zero_grad()
        loss = self.cost_function()
        loss.backward(retain_graph=True)
        return loss
    
    def train(self, epochs, optimizer='Adam', lr=1e-3, weight_decay=1e-5):
        if optimizer == 'Adam':
            self.optimizer = torch.optim.Adam(self.model.parameters(), lr=lr, weight_decay=weight_decay)
        elif optimizer == 'LBFGS':
            self.optimizer = torch.optim.LBFGS(self.model.parameters(), lr=lr, max_iter=20, 
                                             line_search_fn='strong_wolfe')
        
        # Initialize loss history
        loss_keys = self.loss_history.keys()
        for key in loss_keys:
            self.loss_history[key] = np.zeros(epochs)
        
        # Training loop
        for i in range(epochs):
            loss = self.optimizer.step(self.closure)
            
            # Store losses
            for key in loss_keys:
                if key == 'total':
                    self.loss_history[key][i] = loss.item()
                else:
                    self.loss_history[key][i] = self.current_losses[key]
            
            # Print progress
            if i % 100 == 0 or i == epochs - 1:
                print(f"Epoch {i}/{epochs-1}")
                print(f"  Total Loss: {loss.item():.6f}")
                print(f"  PDE Loss: {self.current_losses['pde']:.6f}")
                print(f"  IC Loss: {self.current_losses['ic']:.6f}")
                print(f"  M_out Loss: {self.current_losses['M_out']:.6f}")
                print(f"  M_magnitude Loss: {self.current_losses['M_magnitude']:.6f}")
    
    def plot_loss_history(self):
        # Plot Total Loss
        plt.figure(figsize=(8, 6))
        plt.semilogy(self.loss_history['total'], label='Total Loss')
        plt.xlabel('Epochs', fontsize=14)
        plt.ylabel('Loss (log scale)', fontsize=14)
        plt.title('Total Loss History', fontsize=14)
        plt.legend(fontsize=12)
        plt.tight_layout()
        plt.savefig('total_loss_history.png', dpi=300)
        plt.show()
        
        # Plot PDE Loss
        plt.figure(figsize=(8, 6))
        plt.semilogy(self.loss_history['pde'], label='PDE Loss')
        plt.xlabel('Epochs', fontsize=14)
        plt.ylabel('Loss (log scale)', fontsize=14)
        plt.title('PDE Loss History', fontsize=14)
        plt.legend(fontsize=12)
        plt.tight_layout()
        plt.savefig('pde_loss_history.png', dpi=300)
        plt.show()
        
        # Plot Initial Condition Loss
        plt.figure(figsize=(8, 6))
        plt.semilogy(self.loss_history['ic'], label='IC Loss')
        plt.xlabel('Epochs', fontsize=14)
        plt.ylabel('Loss (log scale)', fontsize=14)
        plt.title('Initial Condition Loss History', fontsize=14)
        plt.legend(fontsize=12)
        plt.tight_layout()
        plt.savefig('ic_loss_history.png', dpi=300)
        plt.show()
    
    def plot_fields_1d(self, t_values, axis='x'):
        """
        Plot 1D slice of fields along a specified axis for multiple time values.
        
        Args:
            t_values (list of float): Times at which to plot the fields
            axis (str): Axis along which to plot ('x', 'y', or 'z')
        """
        n_points = 100
        colors = ['r', 'g', 'b']  # Colors for x, y, z components
        
        if axis == 'x':
            plot_variable = torch.linspace(domain_bounds[0], domain_bounds[1], n_points, dtype=dtype, device=device).view(-1, 1)
            x_test = plot_variable
            y_test = torch.zeros_like(x_test)
            z_test = torch.zeros_like(x_test)
            axis_label = 'x'
        elif axis == 'y':
            plot_variable = torch.linspace(domain_bounds[0], domain_bounds[1], n_points, dtype=dtype, device=device).view(-1, 1)
            y_test = plot_variable
            x_test = torch.zeros_like(y_test)
            z_test = torch.zeros_like(y_test)
            axis_label = 'y'
        else:  # axis == 'z'
            plot_variable = torch.linspace(domain_bounds[0], domain_bounds[1], n_points, dtype=dtype, device=device).view(-1, 1)
            z_test = plot_variable
            x_test = torch.zeros_like(z_test)
            y_test = torch.zeros_like(z_test)
            axis_label = 'z'
        
        plt.figure(figsize=(15, 10))
        
        for t_value in t_values:
            t_test = torch.ones((n_points, 1), dtype=dtype, device=device) * t_value
            
            # Get field values
            E_test, H_test, M_test = self.get_fields(t_test, x_test, y_test, z_test)
            
            # Plot E components
            plt.subplot(3, 1, 1)
            for comp_idx, comp_name in enumerate(['x', 'y', 'z']):
                plt.plot(plot_variable.detach().cpu().numpy(), 
                         E_test[:, comp_idx].detach().cpu().numpy(), 
                         label=f'E_{comp_name} (t={t_value})', color=colors[comp_idx], alpha=0.7)
            plt.xlabel(axis_label)
            plt.ylabel('E')
            plt.title(f'Electric Field Components along {axis_label}')
            plt.legend()
            
            # Plot H components
            plt.subplot(3, 1, 2)
            for comp_idx, comp_name in enumerate(['x', 'y', 'z']):
                plt.plot(plot_variable.detach().cpu().numpy(), 
                         H_test[:, comp_idx].detach().cpu().numpy(), 
                         label=f'H_{comp_name} (t={t_value})', color=colors[comp_idx], alpha=0.7)
            plt.xlabel(axis_label)
            plt.ylabel('H')
            plt.title(f'Magnetic Field Components along {axis_label}')
            plt.legend()
            
            # Plot M components
            plt.subplot(3, 1, 3)
            for comp_idx, comp_name in enumerate(['x', 'y', 'z']):
                plt.plot(plot_variable.detach().cpu().numpy(), 
                         M_test[:, comp_idx].detach().cpu().numpy(), 
                         label=f'M_{comp_name} (t={t_value})', color=colors[comp_idx], alpha=0.7)
            plt.xlabel(axis_label)
            plt.ylabel('M')
            plt.title(f'Magnetization Components along {axis_label}')
            plt.legend()
        
        plt.tight_layout()
        plt.savefig(f'fields_along_{axis}_multiple_times.png', dpi=300)
        plt.show()
    
    def export_to_vtk(self, filename, nx=20, ny=20, nz=20, t_value=None):
        """
        Export the field data to VTK format for ParaView visualization.
        
        Args:
            filename (str): Base filename for the VTK file
            nx, ny, nz (int): Number of grid points in each direction
            t_value (float, optional): Time value for the snapshot. If None, multiple times will be exported.
        """
        # Create output directory
        os.makedirs('vtk_output', exist_ok=True)
        
        # Define times to export
        if t_value is None:
            t_values = np.linspace(0, T, 10)  # Export 10 time steps
        else:
            t_values = [t_value]
        
        # Generate grid
        x = np.linspace(material_bounds[0], material_bounds[1], nx)
        y = np.linspace(material_bounds[0], material_bounds[1], ny)
        z = np.linspace(material_bounds[0], material_bounds[1], nz)
        
        # Create points array for VTK
        points = vtk.vtkPoints()
        
        for k in range(nz):
            for j in range(ny):
                for i in range(nx):
                    points.InsertNextPoint(x[i], y[j], z[k])
        
        # Create a structured grid
        grid = vtk.vtkStructuredGrid()
        grid.SetDimensions(nx, ny, nz)
        grid.SetPoints(points)
        
        # Loop through each time step
        for t_idx, t in enumerate(t_values):
            # Create tensors for grid points
            X, Y, Z = np.meshgrid(x, y, z, indexing='ij')
            grid_points = np.stack([X.flatten(), Y.flatten(), Z.flatten()], axis=1)
            
            t_tensor = torch.ones((grid_points.shape[0], 1), dtype=dtype, device=device) * t
            x_tensor = torch.tensor(grid_points[:, 0].reshape(-1, 1), dtype=dtype, device=device, requires_grad=True)
            y_tensor = torch.tensor(grid_points[:, 1].reshape(-1, 1), dtype=dtype, device=device, requires_grad=True)
            z_tensor = torch.tensor(grid_points[:, 2].reshape(-1, 1), dtype=dtype, device=device, requires_grad=True)
            
            # Get fields
            with torch.no_grad():
                E, H, M = self.get_fields(t_tensor, x_tensor, y_tensor, z_tensor)
            
            # Convert to numpy
            E_np = E.detach().cpu().numpy()
            H_np = H.detach().cpu().numpy()
            M_np = M.detach().cpu().numpy()
            
            # Create VTK arrays for each component
            for field_name, field_data in [('E', E_np), ('H', H_np), ('M', M_np)]:
                for comp_idx, comp_name in enumerate(['x', 'y', 'z']):
                    vtk_array = numpy_support.numpy_to_vtk(field_data[:, comp_idx])
                    vtk_array.SetName(f"{field_name}_{comp_name}")
                    grid.GetPointData().AddArray(vtk_array)
            
            # Create vector arrays
            for field_name, field_data in [('E', E_np), ('H', H_np), ('M', M_np)]:
                vtk_vectors = vtk.vtkDoubleArray()
                vtk_vectors.SetNumberOfComponents(3)
                vtk_vectors.SetName(field_name)
                
                for i in range(field_data.shape[0]):
                    vtk_vectors.InsertNextTuple3(field_data[i, 0], field_data[i, 1], field_data[i, 2])
                
                grid.GetPointData().AddArray(vtk_vectors)
            
            # Calculate field magnitudes
            for field_name, field_data in [('E', E_np), ('H', H_np), ('M', M_np)]:
                magnitude = np.sqrt(field_data[:, 0]**2 + field_data[:, 1]**2 + field_data[:, 2]**2)
                vtk_magnitude = numpy_support.numpy_to_vtk(magnitude)
                vtk_magnitude.SetName(f"{field_name}_magnitude")
                grid.GetPointData().AddArray(vtk_magnitude)
            
            # Add material indicator
            inside_indicator = is_inside(grid_points[:, 0], grid_points[:, 1], grid_points[:, 2]).astype(float)
            vtk_material = numpy_support.numpy_to_vtk(inside_indicator)
            vtk_material.SetName("material")
            grid.GetPointData().AddArray(vtk_material)
            
            # Write the file
            writer = vtk.vtkStructuredGridWriter()
            if len(t_values) > 1:
                output_filename = f"vtk_output/{filename}_t{t_idx:03d}.vtk"
            else:
                output_filename = f"vtk_output/{filename}.vtk"
            
            writer.SetFileName(output_filename)
            writer.SetInputData(grid)
            writer.Write()
            
            print(f"Exported VTK file: {output_filename}")

# ### Main Function
def main():
    """
    Main function to run the PINN-MILLG model training and visualization
    """
    # Create model
    model = PhysicsInformedMILLGModel(t_f, x_f, y_f, z_f, t_ic, x_ic, y_ic, z_ic, 
                                     inside_f_tensor, E0_ic, H0_ic, M0_ic)
    
    # Train model
    print("Starting model training...")
    model.train(epochs=1000, optimizer='Adam', lr=1e-3)
    
    # Switch to L-BFGS for fine-tuning
    print("Fine-tuning with L-BFGS optimizer...")
    model.train(epochs=1000, optimizer='LBFGS', lr=1e-2)
    
    # Plot loss history
    model.plot_loss_history()
    
    # Plot field components
    for t_val in [0.0, 0.05, 0.1]:
        for axis in ['x', 'y', 'z']:
            model.plot_fields_1d([t_val], axis=axis)
    
    # Export to VTK for 3D visualization
    model.export_to_vtk('millg_solution', nx=20, ny=20, nz=20)
    
    print("Simulation and visualization completed.")

if __name__ == "__main__":
    main()